In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import re
import os
import glob

def parse_duration_to_seconds(val):
    """Converts telemetry duration strings (e.g., '16 s', '4 min 56 s') to a float of total seconds."""
    if pd.isna(val):
        return 0.0
    val = str(val).lower().strip()
    minutes, seconds = 0, 0
    
    # Extract minutes if present
    min_match = re.search(r'(\d+)\s*min', val)
    if min_match:
        minutes = int(min_match.group(1))
        
    # Extract seconds if present
    sec_match = re.search(r'(\d+)\s*s', val)
    if sec_match:
        seconds = int(sec_match.group(1))
    elif not min_match:
        # Fallback if it is a plain digit string
        digits = re.findall(r'\d+', val)
        if digits:
            seconds = int(digits[0])
            
    return float((minutes * 60) + seconds)

def run_fleet_analytics(folder_path):
    print("Starting data ingestion and cleaning...")
    
    # Ingest the file (skipping metadata headers if present)
    # Adjust header=1 depending on whether your excel sheets have title blocks
    all_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

    combined_df = []

    for file in all_files:
        try:
            temp_df = pd.read_excel(file, header=1)

            # Optional: track source file
            temp_df['Source_File'] = os.path.basename(file)

            combined_df.append(temp_df)

            print(f"Loaded: {file}")

        except Exception as e:
            print(f"Error loading {file}: {e}")

    # Combine all reports
        if not combined_df:
            print("No Excel reports found.")
            return

    df = pd.concat(combined_df, ignore_index=True)

    print(f"\nTotal records combined: {len(df)}")
    
    # 1. Clean and Transform Data Fields
    df['Speed_kmh'] = pd.to_numeric(
    df['Speed'].astype(str).str.replace(' km/h', '', regex=False),
    errors='coerce')
    df['Duration_seconds'] = df['Alert Duration'].apply(parse_duration_to_seconds)
    df['Alert Starting Time'] = pd.to_datetime(df['Alert Starting Time'])
    df['Hour'] = df['Alert Starting Time'].dt.hour
    df['DayOfWeek'] = df['Alert Starting Time'].dt.day_name()

    # Split telemetry coordinates into dedicated numeric columns
    coords_split = df['Coordinates'].astype(str).str.split('/', expand=True)
    df['Longitude'] = pd.to_numeric(coords_split[0], errors='coerce')
    df['Latitude'] = pd.to_numeric(coords_split[1], errors='coerce')

    # Save fully cleaned data to a master sheet
    df.to_csv('cleaned_fleet_data.csv', index=False)
    print("Saved master cleaned dataset to 'cleaned_fleet_data.csv'")

    # 2. Aggregate Risk Profile Summary
    vehicle_summary = df.groupby('Device Name').agg(
        Total_Alerts=('Alert Type', 'count'),
        True_Violations=('SUMMARY', lambda x: (x =='Valid').sum()),
        False_Alarms=('SUMMARY', lambda x: (x == 'Invalid').sum()),
        Avg_Alert_Speed_kmh=('Speed_kmh', 'mean'),
        Max_Speed_kmh=('Speed_kmh', 'max'),
        Avg_Duration_secs=('Duration_seconds', 'mean')
    ).reset_index()
    vehicle_summary['False_Alarm_Rate_Percent'] = (vehicle_summary['False_Alarms'] / vehicle_summary['Total_Alerts'] * 100).round(1)
    
    vehicle_summary.to_csv('vehicle_risk_profiles.csv', index=False)
    print("Saved aggregated summary profile to 'vehicle_risk_profiles.csv'")

    # 3. Generate Static Trend Visualization - Hourly
    fig1, ax1 = plt.subplots(figsize=(10, 5))
    hourly_counts = df['Hour'].value_counts().reindex(range(24), fill_value=0)

    sns.barplot(x=hourly_counts.index, y=hourly_counts.values, color='dodgerblue', edgecolor='black', ax=ax1)
    ax1.set_title('Fleet Alerts by Hour of Day', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Hour of Day (24-Hour Format)', fontsize=12)
    ax1.set_ylabel('Number of Alerts Triggered', fontsize=12)
    ax1.set_xticks(range(0, 24))
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    fig1.tight_layout()
    fig1.savefig('alerts_by_hour.png')
    plt.close(fig1)

    # 4. Generate Static Trend Visualization - Weekly
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_counts = df['DayOfWeek'].value_counts().reindex(days_order).fillna(0)

    sns.barplot(x=day_counts.index, y=day_counts.values, color='coral', edgecolor='black', ax=ax2)
    ax2.set_title('Fleet Alerts by Day of Week', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Day of Week', fontsize=12)
    ax2.set_ylabel('Number of Alerts Triggered', fontsize=12)
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    fig2.tight_layout()
    fig2.savefig('alerts_by_day.png')
    plt.close(fig2)
    print("Saved static trend visualization charts (.png)")

    # 5. Generate Interactive Geospatial Map (Plotly Express)
    # Formatting richer hover tags for maps
    df['Hover_Text'] = df.apply(
        lambda r: f"Alert: {r['Alert Type']}<br>Status: {r['SUMMARY']}<br>Speed: {r['Speed']}<br>Time: {r['Alert Starting Time']}", 
        axis=1
    )

    fig_map = px.scatter_mapbox(
        df,
        lat="Latitude", 
        lon="Longitude", 
        color="Alert Type", 
        hover_name="Device Name",
        hover_data={"Latitude": False, "Longitude": False, "Hover_Text": True},
        zoom=8, 
        title="Interactive Fleet Alert Hotspot Map"
    )
    fig_map.update_layout(
        mapbox_style="open-street-map",
        margin={"r": 0, "t": 40, "l": 0, "b": 0}
    )
    fig_map.write_html('interactive_fleet_map.html')
    print("Saved zoomable street-view map to 'interactive_fleet_map.html'")
    print("--- Execution Complete ---")

# Execute using your file name
run_fleet_analytics('./fleet_reports')


Starting data ingestion and cleaning...
Loaded: ./fleet_reports\KCG 864S -  04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KCV 947X - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KCZ 474B - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDD 804D - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDN 235V - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDN 563U - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDP 967Z -  04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDS 789W - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDV 176A - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDW 229F - 04.05.2026-10.05.2026.xlsx
Loaded: ./fleet_reports\KDX 834X - 04.05.2026-10.05.2026.xlsx

Total records combined: 2013
Saved master cleaned dataset to 'cleaned_fleet_data.csv'
Saved aggregated summary profile to 'vehicle_risk_profiles.csv'
Saved static trend visualization charts (.png)
Saved zoomable street-view map to 'interactive_fleet_map.html'
--- Execution C

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10576\2168497752.py:134: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(


In [39]:
cleaned_df=pd.read_csv('./cleaned_fleet_data.csv')
cleaned_df.head()

,Device Name,Device ID,Positioning Type,Speed,Alert Starting Time,Alert Ending Time,Alert Duration,Alert Type,Coordinates,SUMMARY,Source_File,Speed_kmh,Duration_seconds,Hour,DayOfWeek,Longitude,Latitude
0,KCG 864S - Mitsubishi FH,2.910723e+11,GPS,46 km/h,2026-05-10 15:48:39,2026-05-10 15:48:55,16 s,Seatbelt Detection,34.736991/-0.828058,Invalid,KCG 864S - 04.05.2026-10.05.2026.xlsx,46.0,16.0,15.0,Sunday,34.736991,-0.828058
1,KCG 864S - Mitsubishi FH,2.910723e+11,GPS,53 km/h,2026-05-10 14:12:37,2026-05-10 14:12:45,8 s,Seatbelt Detection,34.740852/-0.552079,Invalid,KCG 864S - 04.05.2026-10.05.2026.xlsx,53.0,8.0,14.0,Sunday,34.740852,-0.552079
2,KCG 864S - Mitsubishi FH,2.910723e+11,GPS,64 km/h,2026-05-10 13:57:12,2026-05-10 13:57:25,13 s,Seatbelt Detection,34.765152/-0.488466,Invalid,KCG 864S - 04.05.2026-10.05.2026.xlsx,64.0,13.0,13.0,Sunday,34.765152,-0.488466
3,KCG 864S - Mitsubishi FH,2.910723e+11,GPS,38 km/h,2026-05-10 13:52:04,2026-05-10 13:52:07,3 s,Seatbelt Detection,34.801878/-0.484593,Invalid,KCG 864S - 04.05.2026-10.05.2026.xlsx,38.0,3.0,13.0,Sunday,34.801878,-0.484593
4,KCG 864S - Mitsubishi FH,2.910723e+11,GPS,59 km/h,2026-05-10 13:49:07,2026-05-10 13:49:28,21 s,Seatbelt Detection,34.823153/-0.48497,Invalid,KCG 864S - 04.05.2026-10.05.2026.xlsx,59.0,21.0,13.0,Sunday,34.823153,-0.484970


In [40]:
cleaned_df.shape

(2013, 17)

In [41]:
risk_df=pd.read_csv('./vehicle_risk_profiles.csv')
risk_df.head()

,Device Name,Total_Alerts,True_Violations,False_Alarms,Avg_Alert_Speed_kmh,Max_Speed_kmh,Avg_Duration_secs,False_Alarm_Rate_Percent
0,KCG 864S - Mitsubishi FH,457,0,454,53.026258,75.0,15.536105,99.3
1,KCV 947X - Probox,60,0,43,63.700000,97.0,39.283333,71.7
2,KCZ 474B - Toyota Probox,65,0,25,73.076923,91.0,11.338462,38.5
3,KDD 804D - Probox,35,2,27,50.171429,107.0,690.771429,77.1
4,KDN 235V - Ashok Leyland,24,0,24,42.250000,60.0,106.458333,100.0
